# Knee MRI — Standalone M1 Baseline (no GitHub dependency)

A normal, straightforward Kaggle notebook: every cell is plain, self-contained code -- no `git clone`, no importing this project's `src/`, no disk caching, no dataset subsampling. Reads every study, decodes each series' DICOM files fresh every time it's needed, trains, predicts, writes `submission.csv`.

Architecture: the **M1 baseline** from the main project -- ResNet -> masked-mean pooling over a series' slices -> masked-mean pooling over a study's series -> 12 independent sigmoid heads. No slice attention, no series-metadata fusion, no cross-series attention, no pathology adapters, no report supervision -- those live in the full modular repo (`github.com/rebhimohamedamine/RSNA-Knee-Abnormality-Detection`) if you want them later.

**No caching means every epoch re-decodes every DICOM file from scratch.** That's the deliberate trade here -- no disk usage to manage at all (the only things written to `/kaggle/working` are the small model checkpoint and `submission.csv`), at the cost of repeating the full dataset's decode work on every single epoch. `EPOCHS` defaults to 3 below for exactly that reason -- raise it once you've seen how long one epoch actually takes on the full dataset, rather than assuming it'll scale the way a cached run would.

In [ ]:
!pip install -q pydicom opencv-python-headless

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

## Config -- edit these, then re-run every cell below from here down

In [ ]:
DATA_ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
WORK_ROOT = Path('/kaggle/working')
CHECKPOINT_PATH = WORK_ROOT / 'checkpoints' / 'best_model.pt'
SUBMISSION_PATH = WORK_ROOT / 'submission.csv'

LABEL_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
    'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture',
]

IMAGE_SIZE = 128
MAX_SLICES = 16
MAX_SERIES = 6
CLIP_PERCENTILE = (0.5, 99.5)
VAL_FRACTION = 0.2
SPLIT_SEED = 42

BATCH_SIZE = 16
EPOCHS = 3          # no caching -> every epoch re-decodes the whole dataset; raise this only after timing one epoch
LR = 3e-4
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 4
NUM_WORKERS = 4     # DataLoader worker processes -- decoding is CPU-bound, so this is what actually parallelizes it

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## DICOM loading and ordering

In [ ]:
@dataclass
class SliceRecord:
    path: Path
    pixel_array: np.ndarray
    image_position_patient: np.ndarray | None
    image_orientation_patient: np.ndarray | None
    instance_number: int | None
    rescale_slope: float
    rescale_intercept: float


def _to_float_array(value, n):
    if value is None:
        return None
    try:
        arr = np.array([float(v) for v in value], dtype=np.float64)
    except (TypeError, ValueError):
        return None
    return arr if arr.shape == (n,) else None


def load_dicom_series(series_dir: Path) -> list[SliceRecord]:
    records = []
    for path in series_dir.glob('*.dcm'):
        ds = pydicom.dcmread(path)
        ipp = _to_float_array(getattr(ds, 'ImagePositionPatient', None), 3)
        iop = _to_float_array(getattr(ds, 'ImageOrientationPatient', None), 6)
        instance_number = getattr(ds, 'InstanceNumber', None)
        records.append(SliceRecord(
            path=path, pixel_array=ds.pixel_array,
            image_position_patient=ipp, image_orientation_patient=iop,
            instance_number=int(instance_number) if instance_number is not None else None,
            rescale_slope=float(getattr(ds, 'RescaleSlope', 1.0)),
            rescale_intercept=float(getattr(ds, 'RescaleIntercept', 0.0)),
        ))
    return records


def order_slices(records: list[SliceRecord]) -> list[SliceRecord]:
    """ImagePositionPatient projected onto the plane normal (primary),
    InstanceNumber (fallback), filename (last resort). A condensed version
    of the full repo's fallback chain -- good enough here; that version also
    falls back to SliceLocation before filename."""
    if len(records) <= 1:
        return list(records)
    if all(r.image_position_patient is not None and r.image_orientation_patient is not None for r in records):
        row, col = records[0].image_orientation_patient[0:3], records[0].image_orientation_patient[3:6]
        normal = np.cross(row, col)
        normal = normal / (np.linalg.norm(normal) + 1e-8)
        return sorted(records, key=lambda r: float(np.dot(r.image_position_patient, normal)))
    if all(r.instance_number is not None for r in records):
        return sorted(records, key=lambda r: r.instance_number)
    return sorted(records, key=lambda r: r.path.name)


def read_pixel_array(record: SliceRecord) -> np.ndarray:
    return record.pixel_array.astype(np.float32) * record.rescale_slope + record.rescale_intercept

## Preprocessing and slice sampling

In [ ]:
def clip_and_scale(img: np.ndarray) -> np.ndarray:
    lo, hi = np.percentile(img, CLIP_PERCENTILE)
    span = hi - lo
    if span < 1e-6:
        return np.zeros_like(img, dtype=np.float32)
    clipped = np.clip(img, lo, hi).astype(np.float64)
    return np.clip((clipped - lo) / span, 0.0, 1.0).astype(np.float32)


def resize_slice(img: np.ndarray) -> np.ndarray:
    return cv2.resize(img.astype(np.float32), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)


def select_uniform_indices(n_available: int, k: int) -> list[int]:
    if k <= 0 or n_available <= 0:
        return []
    if n_available <= k:
        return list(range(n_available))
    if k == 1:
        return [n_available // 2]
    return sorted(set(int(round(x)) for x in np.linspace(0, n_available - 1, k)))


def build_series_array(series_dir: Path) -> np.ndarray:
    """DICOM directory -> (n_selected, IMAGE_SIZE, IMAGE_SIZE) float32 in [0,1].
    Called fresh every time a series is needed -- nothing is cached to disk."""
    records = order_slices(load_dicom_series(series_dir))
    slices = [resize_slice(clip_and_scale(read_pixel_array(r))) for r in records]
    if not slices:
        return np.zeros((0, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
    indices = select_uniform_indices(len(slices), MAX_SLICES)
    return np.stack([slices[i] for i in indices], axis=0)

## Study-level split (never leak a study's series across train/val)

In [ ]:
def split_studies(studies_df: pd.DataFrame, val_fraction: float, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    uids = studies_df['StudyInstanceUID']
    if uids.duplicated().any():
        raise ValueError('Duplicate StudyInstanceUID -- would leak a study across train/val')
    rng = np.random.RandomState(seed)
    order = np.arange(len(studies_df))
    rng.shuffle(order)
    n_val = int(round(len(order) * val_fraction))
    val_idx, train_idx = order[:n_val], order[n_val:]
    return studies_df.iloc[train_idx].reset_index(drop=True), studies_df.iloc[val_idx].reset_index(drop=True)

## Dataset

In [ ]:
class KneeBaselineDataset(Dataset):
    def __init__(self, studies_df: pd.DataFrame, series_df: pd.DataFrame, dicom_root: Path, has_labels: bool):
        self.studies_df = studies_df.reset_index(drop=True)
        self.dicom_root = dicom_root
        self.has_labels = has_labels
        self.series_by_study = {uid: df for uid, df in series_df.groupby('StudyInstanceUID')}

    def __len__(self):
        return len(self.studies_df)

    def __getitem__(self, idx):
        row = self.studies_df.iloc[idx]
        study_uid = str(row['StudyInstanceUID'])
        series_rows = self.series_by_study.get(study_uid)
        series_uids = [] if series_rows is None else list(series_rows['SeriesInstanceUID'])[:MAX_SERIES]

        pixel_values = np.zeros((MAX_SERIES, MAX_SLICES, 1, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        slice_mask = np.zeros((MAX_SERIES, MAX_SLICES), dtype=bool)
        series_mask = np.zeros((MAX_SERIES,), dtype=bool)

        for s, series_uid in enumerate(series_uids):
            series_uid = str(series_uid)
            series_dir = self.dicom_root / study_uid / series_uid
            if not series_dir.exists():
                continue
            array = build_series_array(series_dir)
            n = array.shape[0]
            if n > 0:
                pixel_values[s, :n, 0] = array
                slice_mask[s, :n] = True
            series_mask[s] = True

        if self.has_labels:
            labels_raw = row[LABEL_COLUMNS].to_numpy(dtype=np.float32)
            label_mask = (~np.isnan(labels_raw)).astype(np.float32)
            labels = np.nan_to_num(labels_raw, nan=0.0).astype(np.float32)
        else:
            labels = np.zeros(len(LABEL_COLUMNS), dtype=np.float32)
            label_mask = np.zeros(len(LABEL_COLUMNS), dtype=np.float32)

        return {
            'pixel_values': torch.from_numpy(pixel_values),
            'slice_mask': torch.from_numpy(slice_mask),
            'series_mask': torch.from_numpy(series_mask),
            'labels': torch.from_numpy(labels),
            'label_mask': torch.from_numpy(label_mask),
            'study_uid': study_uid,
        }

## Model -- M1: ResNet + masked-mean pooling (twice) + 12 heads

In [ ]:
class KneeBaselineModel(nn.Module):
    def __init__(self, embedding_dim: int = 128, backbone: str = 'resnet18', pretrained: bool = False):
        super().__init__()
        ctor, feat_dim = {'resnet18': (tv_models.resnet18, 512), 'resnet34': (tv_models.resnet34, 512)}[backbone]
        net = ctor(weights='IMAGENET1K_V1' if pretrained else None)
        old_conv = net.conv1
        net.conv1 = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size, stride=old_conv.stride, padding=old_conv.padding, bias=False)
        net.fc = nn.Identity()
        self.backbone = net
        self.projection = nn.Linear(feat_dim, embedding_dim)
        self.heads = nn.ModuleDict({label: nn.Linear(embedding_dim, 1) for label in LABEL_COLUMNS})

    def forward(self, pixel_values, slice_mask, series_mask):
        B, S, K, C, H, W = pixel_values.shape
        series_embeddings = []
        for s in range(S):
            slices = pixel_values[:, s].reshape(B * K, C, H, W)
            feats = self.projection(self.backbone(slices)).view(B, K, -1)
            mask = slice_mask[:, s].float().unsqueeze(-1)
            denom = mask.sum(dim=1).clamp_min(1.0)
            pooled = (feats * mask).sum(dim=1) / denom
            series_embeddings.append(pooled)
            del slices, feats
        series_embeddings = torch.stack(series_embeddings, dim=1)

        smask = series_mask.float().unsqueeze(-1)
        sdenom = smask.sum(dim=1).clamp_min(1.0)
        study_embedding = (series_embeddings * smask).sum(dim=1) / sdenom

        return torch.cat([self.heads[label](study_embedding) for label in LABEL_COLUMNS], dim=-1)

## Loss, metrics, train/eval/inference loops

In [ ]:
def masked_bce_loss(logits, labels, label_mask):
    per_element = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
    return (per_element * label_mask).sum() / label_mask.sum().clamp_min(1e-8)


def compute_macro_auc(y_true: np.ndarray, y_prob: np.ndarray, label_mask: np.ndarray) -> tuple[float, dict]:
    per_target = {}
    for i, name in enumerate(LABEL_COLUMNS):
        mask = label_mask[:, i].astype(bool)
        y, p = y_true[mask, i], y_prob[mask, i]
        if mask.sum() < 2 or len(np.unique(y)) < 2:
            per_target[name] = float('nan')
            continue
        per_target[name] = float(roc_auc_score(y, p))
    values = np.array(list(per_target.values()))
    macro = float(np.nanmean(values)) if np.any(~np.isnan(values)) else float('nan')
    return macro, per_target


def move_batch(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch['pixel_values'], batch['slice_mask'], batch['series_mask'])
        loss = masked_bce_loss(logits, batch['labels'], batch['label_mask'])
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(1, n_batches)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_labels, all_probs, all_masks = [], [], []
    for batch in loader:
        batch = move_batch(batch, device)
        logits = model(batch['pixel_values'], batch['slice_mask'], batch['series_mask'])
        all_labels.append(batch['labels'].cpu().numpy())
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_masks.append(batch['label_mask'].cpu().numpy())
    y_true = np.concatenate(all_labels, axis=0)
    y_prob = np.concatenate(all_probs, axis=0)
    label_mask = np.concatenate(all_masks, axis=0)
    return compute_macro_auc(y_true, y_prob, label_mask)


@torch.no_grad()
def run_inference(model, loader, device) -> pd.DataFrame:
    model.eval()
    study_uids, all_probs = [], []
    for batch in loader:
        batch = move_batch(batch, device)
        logits = model(batch['pixel_values'], batch['slice_mask'], batch['series_mask'])
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        study_uids.extend(batch['study_uid'])
    probs = np.concatenate(all_probs, axis=0)
    df = pd.DataFrame(probs, columns=LABEL_COLUMNS)
    df.insert(0, 'StudyInstanceUID', study_uids)
    return df


def write_submission(pred_df: pd.DataFrame, sample_submission_path: Path, out_path: Path) -> None:
    sample = pd.read_csv(sample_submission_path)
    columns = list(sample.columns)
    pred_indexed = pred_df.set_index('StudyInstanceUID')
    rows = []
    for study_uid in sample['StudyInstanceUID']:
        if study_uid in pred_indexed.index:
            row = pred_indexed.loc[study_uid]
            rows.append([study_uid] + [float(row[c]) for c in columns[1:]])
        else:
            rows.append([study_uid] + [0.5] * (len(columns) - 1))
    out_df = pd.DataFrame(rows, columns=columns)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(f'Wrote {len(out_df)} rows to {out_path}')

## Run it: load data, split, train (full training set, no subsampling)

In [ ]:
studies_df = pd.read_csv(DATA_ROOT / 'train.csv')
series_df = pd.read_csv(DATA_ROOT / 'train_series.csv')
print(f'{len(studies_df)} studies / {len(series_df)} series total -- using all of them')

train_df, val_df = split_studies(studies_df, VAL_FRACTION, SPLIT_SEED)
print(f'train/val studies: {len(train_df)}/{len(val_df)}')

train_ds = KneeBaselineDataset(train_df, series_df, DATA_ROOT / 'train_series', has_labels=True)
val_ds = KneeBaselineDataset(val_df, series_df, DATA_ROOT / 'train_series', has_labels=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
model = KneeBaselineModel(embedding_dim=128, backbone='resnet18', pretrained=False).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_score = -float('inf')
epochs_without_improvement = 0
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    macro_auc, per_target_auc = evaluate(model, val_loader, DEVICE)
    score = macro_auc if not np.isnan(macro_auc) else -float('inf')
    print(f'epoch {epoch}: train_loss={train_loss:.4f} val_macro_auc={macro_auc}')

    improved = score > best_score or epoch == 0
    if improved:
        best_score = max(best_score, score)
        torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_macro_auc': macro_auc, 'per_target_auc': per_target_auc}, CHECKPOINT_PATH)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f'Early stopping at epoch {epoch} (no improvement for {EARLY_STOPPING_PATIENCE} epochs).')
            break

print('Best checkpoint:', CHECKPOINT_PATH)

## Inference on the test set -> submission.csv

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val macro AUC = {checkpoint['val_macro_auc']})")

test_studies_df = pd.read_csv(DATA_ROOT / 'test.csv')
test_series_df = pd.read_csv(DATA_ROOT / 'test_series.csv')
for col in LABEL_COLUMNS:
    if col not in test_studies_df.columns:
        test_studies_df[col] = float('nan')  # the real test set has no ground truth -- keeps the Dataset contract simple

test_ds = KneeBaselineDataset(test_studies_df, test_series_df, DATA_ROOT / 'test_series', has_labels=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

pred_df = run_inference(model, test_loader, DEVICE)
write_submission(pred_df, DATA_ROOT / 'sample_submission.csv', SUBMISSION_PATH)
pred_df.head()

## Notes

- **This is M1 only.** Slice attention, series-metadata fusion, cross-series attention, pathology-group adapters, and report weak-supervision all exist in the full repo (`github.com/rebhimohamedamine/RSNA-Knee-Abnormality-Detection`) but are deliberately not inlined here -- adding them would mean re-deriving most of that repo's `src/models/`, `src/reports/`, and `src/training/` inside notebook cells.
- **No caching, no study cap, by design** -- every epoch decodes the full training set's DICOM files from scratch, and nothing beyond the checkpoint and `submission.csv` is written to `/kaggle/working`, so there's no disk quota to manage at all. The trade-off is time, not disk: watch how long epoch 0 actually takes before assuming `EPOCHS` can go higher, since that cost repeats identically on every epoch (there's no "faster after the first pass" the way a cached run has).
- If per-epoch time turns out to be the bottleneck, the fix is caching (`notebooks/kaggle_standalone_baseline.ipynb`'s previous version had this, or ask for it back) or a study subsample (`data.max_studies` in the main repo's configs) -- both trade disk for time, in case that trade is worth revisiting once you've seen real numbers.
- **Reproducibility**: `SPLIT_SEED` controls the train/val split -- keep it fixed to compare runs meaningfully.